# Smart Wastewater System - ML Pipeline Training
This notebook trains your models in the cloud. Run each cell sequentially.

In [1]:
!pip install pandas numpy scikit-learn tensorflow matplotlib joblib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
# Create directories if they don't exist
os.makedirs('dataset', exist_ok=True)
os.makedirs('models', exist_ok=True)

## 1. Upload Dataset
Upload `dataset/pollution_data.csv` to to the Colab `dataset/` folder, OR uncomment the cell below to generate it instantly on Colab.

In [3]:
import numpy as np
import pandas as pd
import random
samples = 10000
data = []
for i in range(samples):
    label = random.choice([0,1,2])
    if label == 0:
        ph = np.random.uniform(6.5,7.5)
        tds = np.random.uniform(200,400)
        turbidity = np.random.uniform(0,50)
        orp = np.random.uniform(300,400)
        temp = np.random.uniform(25,30)
    elif label == 1:
        ph = np.random.uniform(6.0,7.0)
        tds = np.random.uniform(500,900)
        turbidity = np.random.uniform(100,300)
        orp = np.random.uniform(250,350)
        temp = np.random.uniform(25,32)
    else:
        ph = np.random.uniform(5.5,6.5)
        tds = np.random.uniform(350,600)
        turbidity = np.random.uniform(50,120)
        orp = np.random.uniform(150,300)
        temp = np.random.uniform(25,30)
    data.append([ph,tds,turbidity,orp,temp,label])

df = pd.DataFrame(data, columns=["ph","tds","turbidity","orp","temperature","label"])
df.to_csv("dataset/pollution_data.csv", index=False)
print("Dataset created.")

Dataset created.


## 2. Train Random Forest Classifier

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

data = pd.read_csv("dataset/pollution_data.csv")
X = data[['ph','tds','turbidity','orp','temperature']]
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train,y_train)
pred = model.predict(X_test)

print(classification_report(y_test,pred))
joblib.dump(model,"models/pollution_classifier.pkl")
print("Random Forest Model Saved.")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       670
           1       1.00      0.99      1.00       659
           2       0.99      1.00      1.00       671

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Random Forest Model Saved.


## 3. Train LSTM Model

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

features = data[['ph','tds','turbidity','orp','temperature']].values
scaler = MinMaxScaler()
features = scaler.fit_transform(features)

joblib.dump(scaler, "models/scaler.pkl") # Required for inference

sequence_length = 10
X = []
y = []

for i in range(len(features)-sequence_length):
    X.append(features[i:i+sequence_length])
    y.append(features[i+sequence_length][2]) # predicting turbidity

X = np.array(X)
y = np.array(y)

lstm_model = Sequential()
lstm_model.add(LSTM(64,input_shape=(sequence_length,5)))
lstm_model.add(Dense(32,activation='relu'))
lstm_model.add(Dense(1))

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X,y,epochs=10,batch_size=32)
lstm_model.save("models/lstm_model.h5")
print("LSTM Model Saved.")

c:\Users\ryang\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0764
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0748
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0741
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0740
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0740
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0739
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0739
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0740
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0737
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0738


LSTM Model Saved.


## 4. Download Models Locally
Right click the `models` folder on the left pane and download `pollution_classifier.pkl`, `scaler.pkl`, and `lstm_model.h5` locally into your project.